# Download Production Models from DagShub

Downloads `@prod` weights for the Start/Stop predictor and Good/Bad classifier and saves them locally as `.pth` + `.json` config files.

In [1]:
import os, json
import torch
import mlflow
import mlflow.pytorch
import dagshub
import joblib

dagshub.init(repo_owner='SamuelFredricBerg', repo_name='4dt907', mlflow=True)
client = mlflow.MlflowClient()
device = torch.device('cpu')
print('Connected to DagShub')


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=311cd882-00f6-4af4-8915-0ab83c0a5323&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=910d11e81b8cb149ecb279eadf9d4fb72a8cdfbe4edf387ead6f06c367b4dbe4




Accessing as Soppa22

Initialized MLflow to track repo "SamuelFredricBerg/4dt907"

Repository SamuelFredricBerg/4dt907 initialized!

Connected to DagShub


In [2]:
def download_model(project_name, alias, pth_name, config_name, scaler_name):
    """
    Downloads a registered model @alias, saves:
      - weights  → pth_name
      - config   → config_name  (all run params)
      - scaler   → scaler_name  (if logged)
    """
    version = client.get_model_version_by_alias(project_name, alias)
    run     = client.get_run(version.run_id)
    params  = run.data.params

    print(f'\n{project_name} @{alias}')
    print(f'  run_id  : {version.run_id[:12]}')
    print(f'  version : {version.version}')
    print(f'  params  : {params}')

    # Load model and save state dict
    model = mlflow.pytorch.load_model(
        f'models:/{project_name}@{alias}', map_location=device
    )
    torch.save(model.state_dict(), pth_name)
    print(f'  Saved weights -> {pth_name}')

    # Save params as config json
    with open(config_name, 'w') as f:
        json.dump(params, f, indent=2)
    print(f'  Saved config  -> {config_name}')

    # Save scaler if one was logged
    artifacts = [a.path for a in client.list_artifacts(version.run_id)]
    scaler_artifact = next(
        (a for a in artifacts if 'scaler' in a.lower() and a.endswith('.joblib')), None
    )
    if scaler_artifact:
        local_path = client.download_artifacts(version.run_id, scaler_artifact)
        scaler = joblib.load(local_path)
        joblib.dump(scaler, scaler_name)
        print(f'  Saved scaler  -> {scaler_name}')
    else:
        print(f'  No scaler found')

    return params


## Start/Stop Predictor

In [3]:
SS_PROJECT = 'Start_Stop_Predictor_ModelV2'

ss_params = download_model(
    project_name = SS_PROJECT,
    alias        = 'prod',
    pth_name     = 'best_start_stop_model_prod.pth',
    config_name  = 'start_stop_config.json',
    scaler_name  = 'scaler_start_stop.joblib',
)



Start_Stop_Predictor_ModelV2 @prod
  run_id  : 4380b764a18c
  version : 6
  params  : {'note': 'LSTM_Dense_Uni', 'seed': '42', 'seq_length': '5', 'hidden_size': '128', 'lr': '0.0005', 'batch_size': '64', 'epochs': '30', 'patience': '5', 'num_layers': '2', 'dropout': '0.1', 'input_size': '39', 'use_scaling': 'False', 'activation': 'identity', 'init': 'default', 'bidirectional': 'False', 'rnn_type': 'LSTM'}


  Saved weights -> best_start_stop_model_prod.pth
  Saved config  -> start_stop_config.json
  No scaler found


## Good/Bad Classifier

In [4]:
# ✏️ Change this if your registry name differs
GB_PROJECT = 'GoodBad_ClassifierV2'

gb_params = download_model(
    project_name = GB_PROJECT,
    alias        = 'prod',
    pth_name     = 'best_goodbad_model_prod.pth',
    config_name  = 'goodbad_config.json',
    scaler_name  = 'scaler_goodbad_prod.joblib',
)



GoodBad_ClassifierV2 @prod
  run_id  : 9bc80f23bc30
  version : 7
  params  : {'note': 'ACNN_v1', 'model_type': 'cnn', 'n_filters': '8', 'kernel_size': '3', 'pool_size': '2', 'fc_h': '16', 'dropout': '0.3', 'lr': '0.001', 'batch_size': '64', 'epochs': '80', 'patience': '10', 'use_scaling': 'True', 'c_frames': '10', 'n_features': '61', 'input_dim': '610', 'n_params': '2145'}


  Saved weights -> best_goodbad_model_prod.pth
  Saved config  -> goodbad_config.json


  Saved scaler  -> scaler_goodbad_prod.joblib


## Verify Downloads

In [5]:
import os

files = [
    'best_start_stop_model_prod.pth',
    'start_stop_config.json',
    'best_goodbad_model_prod.pth',
    'goodbad_config.json',
]

print('Downloaded files:')
for f in files:
    size = os.path.getsize(f) if os.path.exists(f) else None
    status = f'{size:,} bytes' if size else 'MISSING'
    print(f'  {f:<45} {status}')

print('\nStart/stop config:')
with open('start_stop_config.json') as f:
    print(json.dumps(json.load(f), indent=2))

print('\nGood/bad config:')
with open('goodbad_config.json') as f:
    print(json.dumps(json.load(f), indent=2))


Downloaded files:
  best_start_stop_model_prod.pth                879,042 bytes
  start_stop_config.json                        356 bytes
  best_goodbad_model_prod.pth                   11,496 bytes
  goodbad_config.json                           332 bytes

Start/stop config:
{
  "note": "LSTM_Dense_Uni",
  "seed": "42",
  "seq_length": "5",
  "hidden_size": "128",
  "lr": "0.0005",
  "batch_size": "64",
  "epochs": "30",
  "patience": "5",
  "num_layers": "2",
  "dropout": "0.1",
  "input_size": "39",
  "use_scaling": "False",
  "activation": "identity",
  "init": "default",
  "bidirectional": "False",
  "rnn_type": "LSTM"
}

Good/bad config:
{
  "note": "ACNN_v1",
  "model_type": "cnn",
  "n_filters": "8",
  "kernel_size": "3",
  "pool_size": "2",
  "fc_h": "16",
  "dropout": "0.3",
  "lr": "0.001",
  "batch_size": "64",
  "epochs": "80",
  "patience": "10",
  "use_scaling": "True",
  "c_frames": "10",
  "n_features": "61",
  "input_dim": "610",
  "n_params": "2145"
}
